In [44]:
import pandas as pd

In [45]:
# df_conversations = pd.read_parquet('conversations.parquet',engine='fastparquet')
df_reactions = pd.read_parquet('reactions.parquet', engine='pyarrow')
# df_votes = pd.read_parquet('votes.parquet',engine='fastparquet')

In [ ]:
# print(df_conversations.columns)
print('\n')
print(df_reactions.columns)
print('\n')
# print(df_votes.columns)




Index(['id', 'timestamp', 'model_a_name', 'model_b_name', 'conversation_a',
       'conversation_b', 'conv_turns', 'conversation_pair_id', 'conv_a_id',
       'conv_b_id', 'session_hash', 'visitor_id', 'model_pair_name',
       'opening_msg', 'system_prompt_a', 'system_prompt_b', 'mode',
       'custom_models_selection', 'short_summary', 'keywords', 'categories',
       'languages', 'total_conv_a_output_tokens', 'total_conv_b_output_tokens',
       'model_a_total_params', 'model_b_total_params', 'model_a_active_params',
       'model_b_active_params', 'total_conv_a_kwh', 'total_conv_b_kwh'],
      dtype='str')


Index(['id', 'timestamp', 'model_a_name', 'model_b_name', 'refers_to_model',
       'msg_index', 'opening_msg', 'conversation_a', 'conversation_b',
       'model_pos', 'conv_turns', 'conversation_pair_id', 'conv_a_id',
       'conv_b_id', 'refers_to_conv_id', 'session_hash', 'visitor_id',
       'response_content', 'question_content', 'liked', 'disliked', 'comment',
       'use

In [47]:
df_reactions.iloc[60]


id                                                                      176493
timestamp                                           2025-03-25 13:49:04.671683
model_a_name                                     deepseek-r1-distill-llama-70b
model_b_name                                                           qwq-32b
refers_to_model                                  deepseek-r1-distill-llama-70b
msg_index                                                                    1
opening_msg                      bonjour, quel est le meilleur langage de prog
conversation_a               [{'content': 'bonjour, quel est le meilleur la...
conversation_b               [{'content': 'bonjour, quel est le meilleur la...
model_pos                                                                    a
conv_turns                                                                   1
conversation_pair_id         0f139d2e4b16403d8ecae736b60ca3cc-79ef955c54494...
conv_a_id                                     0f139d

In [48]:
print(len(df_reactions))

94294


In [49]:
df_reactions = df_reactions[:100]

In [50]:
df_reactions.response_content.str.len().mean()

np.float64(2503.99)

# 8. Question 4 — Validité de construit 

In [91]:
! pip install sentence_transformers

In [92]:
### Implémentation de MATTR
from collections import Counter
import numpy as np
import spacy
from sentence_transformers import SentenceTransformer
import torch
from torch.nn.functional import cosine_similarity


# Longformer (jusqu'à 4096 tokens)
model_name = "allenai/longformer-base-4096"

# Ou sentence-transformers qui gère ça automatiquement
model = SentenceTransformer("dangvantuan/sentence-camembert-large")


nlp = spacy.load("fr_core_news_sm")
model_name = "camembert-base"

def tokenize(text):
    doc = nlp(text)
    tokens = [token.text for token in doc]
    return tokens

def moving_average_ttr(tokens,window_size=100):
    if len(tokens)==0:
        return 0
    if len(tokens) < window_size:
        return len(set(tokens)) / len(tokens)
    
    
    ttr_list = []
    n_chunk = len(tokens) // window_size
    
    for i in range(n_chunk):
        window = tokens[i*window_size : (i+1)*window_size]
        ttr_list.append(len(set(window)) / len(window))
    
    if len(tokens) % window_size != 0:
        last_window = tokens[n_chunk*window_size:]
        ttr_list.append(len(set(last_window)) / len(last_window))
    
    return np.mean(ttr_list)

### Implémentation du N-gramme Rarity Score
def corpus_to_Ngram(corpus,N=2):

    Ngram_corpus = [
        " ".join(token_list[i:i+N])
        for token_list in corpus
        for i in range(len(token_list)-N+1)
    ]
    return set(Ngram_corpus)


def N_gram_rarity(token,vocab_corpus,N=2):
    if len(token)==0:
        return 0
    if len(token) < N:
        Ngram = [" ".join(token)]
    Ngram = [" ".join(token[i:i+N]) for i in range(len(token)-N+1)]
    vocab = set(Ngram)

    return len(vocab - vocab_corpus)/len(vocab)

def get_embedding(text):
    embedding = model.encode(text)
    return embedding

def get_corpus_mean_embedding_np(corpus):
    embeddings = [get_embedding(text) for text in corpus]
    embeddings = torch.cat(embeddings, dim=0)
    mean_embedding = embeddings.mean(dim=0)
    return mean_embedding


def embedding_distance(v1, v2):
    v1 = v1.squeeze().unsqueeze(0)    # (1, D)
    v2 = v2.squeeze().unsqueeze(0)  # (1, D)
    
    similarity = cosine_similarity(v1, v2)  # dim=1 par défaut
    return 1.0 - similarity.item()

No sentence-transformers model found with name dangvantuan/sentence-camembert-large. Creating a new one with mean pooling.


config.json:   0%|          | 0.00/683 [00:00<?, ?B/s]

c:\Users\youce\miniconda3\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\youce\.cache\huggingface\hub\models--dangvantuan--sentence-camembert-large. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/1.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/400 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/809k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/298 [00:00<?, ?B/s]

Could not extract SentencePiece model from C:\Users\youce\.cache\huggingface\hub\models--dangvantuan--sentence-camembert-large\snapshots\1f111fd01a1c8595a4e8478775d740ab03279507\sentencepiece.bpe.model using sentencepiece library due to 
SentencePieceExtractor requires the SentencePiece library but it was not found in your environment. Check out the instructions on the
installation page of its repo: https://github.com/google/sentencepiece#installation and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.
. Falling back to TikToken extractor.


ValueError: `tiktoken` is required to read a `tiktoken` file. Install it with `pip install tiktoken`.

In [ ]:
df_reactions['tokens'] = df_reactions.response_content.apply(lambda x: tokenize(x))

### MATTR

In [ ]:
df_reactions['MATTR'] = df_reactions.tokens.apply(lambda x: moving_average_ttr(x))

### N-gram rarity

In [ ]:
! pip install datasets

In [ ]:
! pip install "datasets<2.20"

In [ ]:
from datasets import load_dataset
from collections import Counter

def build_ngram_vocab_fast(dataset, nlp, N=2, limit=50000, batch_size=1000):
    counter = Counter()
    texts = []

    for i, x in enumerate(dataset):
        texts.append(x["text"])

        if len(texts) == batch_size:
            for doc in nlp.pipe(texts):
                tokens = [t.text.lower() for t in doc if t.is_alpha]
                ngrams = [" ".join(tokens[j:j+N]) for j in range(len(tokens)-N+1)]
                counter.update(ngrams)
            texts = []

        if i >= limit:
            break

    # Traiter le reste
    if texts:
        for doc in nlp.pipe(texts):
            tokens = [t.text.lower() for t in doc if t.is_alpha]
            ngrams = [" ".join(tokens[j:j+N]) for j in range(len(tokens)-N+1)]
            counter.update(ngrams)

    return set(counter)


# Charger et filtrer OASST1
ds = load_dataset("OpenAssistant/oasst1")
ds_fr = ds.filter(lambda x: x["lang"] == "fr")

# Utiliser le split "train" explicitement
dataset_fr = ds_fr["train"]

print(f"Nombre d'exemples FR (train) : {len(dataset_fr)}")
print("Colonnes disponibles :", dataset_fr.column_names)

# Construire les n-grammes
ngram_freq = build_ngram_vocab_fast(dataset_fr, nlp, N=2, limit=50000)

# print(f"\nTop 20 bigrammes :")
# for ngram, count in ngram_freq.most_common(20):
#     print(f"  {ngram}: {count}")

Nombre d'exemples FR (train) : 2474
Colonnes disponibles : ['message_id', 'parent_id', 'user_id', 'created_date', 'text', 'role', 'lang', 'review_count', 'review_result', 'deleted', 'rank', 'synthetic', 'model_name', 'detoxify', 'message_tree_id', 'tree_state', 'emojis', 'labels']


In [ ]:
df_reactions['N_gram_rarity'] = df_reactions.tokens.apply(lambda x: N_gram_rarity(x,ngram_freq))

### Embedding distance

In [ ]:
response = df_reactions.iloc[0].response_content
question = df_reactions.iloc[0].question_content
print(get_embedding(response))

RuntimeError: index 514 is out of bounds for dimension 1 with size 514

In [ ]:
def embedding_row(row):
    print(row.response_content)
    print(row.question_content)
    response = get_embedding(row.response_content)
    print(response.shape)
    question = get_embedding(row.question_content)
    print(question.shape)
    return embedding_distance(response,question)

df_reactions['embedding_distance']= df_reactions.apply(embedding_row,axis=1)

La stratégie commerciale de **Champagne Nicolas Feuillatte** s'articule autour de plusieurs axes clés pour renforcer sa position sur le marché des champagnes, en particulier en tant que **marque coopérative leader** en volume. Voici les éléments structurants de leur approche :

### 1. **Positionnement et Identité**  
   - **Accessibilité premium** : Nicolas Feuillatte se positionne comme une marque **accessible** au sein du segment premium, avec des prix compétitifs par rapport aux grandes maisons traditionnelles (ex. Moët & Chandon).  
   - **Image moderne et élégante** : Design épuré des bouteilles, communication axée sur le **luxe abordable**, pour attirer une clientèle jeune et internationale.  
   - **Mise en avant de la coopérative** : Valorisation des **5 000 vignerons** membres, avec un discours sur l'authenticité et le terroir de la Champagne.

### 2. **Distribution et Marchés Cibles**  
   - **Réseau international** : Présence dans **100 pays**, avec une forte dépendance à l'

RuntimeError: index 514 is out of bounds for dimension 1 with size 514

In [ ]:
corr_mattr = df_reactions.creative.corr(df_reactions.MATTR)
corr_ngram = df_reactions.creative.corr(df_reactions.N_gram_rarity)
corr_embedding = df_reactions.creative.corr(df_reactions.embedding_distance)


print(corr_mattr)
print(corr_ngram)
print(corr_embedding)